In [ ]:
# --- Clean install for a fresh Colab runtime (Python 3.12) ---

# first select GPU under runtime

# 1) Upgrade installer tooling
!pip -q install -U pip setuptools wheel

# 2) Pin a stable NumPy range (avoids NumPy 2.4.x conflicts, keeps OpenCV happy)
!pip -q install "numpy>=2.1,<2.2"

# 3) Scientific stack compatible with NumPy 2.1 on Py3.12
!pip -q install "scipy>=1.13" matplotlib

# 4) OpenCV (headless is best for training)
!pip -q install "opencv-python-headless>=4.9.0.80"

# 5) SAM + HF + data + losses
!pip -q install git+https://github.com/facebookresearch/segment-anything.git
!pip -q install git+https://github.com/huggingface/transformers.git
!pip -q install datasets monai tifffile

# 6) patchify WITHOUT dependencies so it doesn't force numpy<2
!pip -q install --no-deps patchify


In [ ]:
#importing all the packages and look at versions

import numpy as np, scipy, cv2
from scipy import ndimage
from patchify import patchify
import tifffile
import matplotlib.pyplot as plt

print("numpy:", np.__version__)
print("scipy:", scipy.__version__)
print("opencv:", cv2.__version__)
print("all imports ok")


In [ ]:
#check pytorch version

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


In [ ]:
#connect to google drive where the folders are

from google.colab import drive
drive.mount("/content/drive")


In [ ]:
#set paths for training data

images_path = "/content/drive/MyDrive/Colab/microscopy.tif"
masks_path  = "/content/drive/MyDrive/Colab/masks.tif"


In [ ]:
#check image sizes

import tifffile
import os

print("images exists:", os.path.exists(images_path))
print("masks exists:", os.path.exists(masks_path))

imgs = tifffile.imread(images_path)
msks = tifffile.imread(masks_path)

print("imgs shape:", imgs.shape, imgs.dtype)
print("msks shape:", msks.shape, msks.dtype)


In [ ]:
import numpy as np
print("mask unique values (sample):", np.unique(msks[0])[:20])
print("mask max:", msks.max())

#gotta make it binary masks if it is [0 1 2]

In [ ]:
#binarize mask

msks_bin = (msks > 0).astype(np.uint8)  # 0/1
print("binary unique:", np.unique(msks_bin))


In [ ]:
# creating patches 256

import numpy as np
from patchify import patchify

patch_size = 256
step = 256  # no overlap

# --- Images (RGB) ---
all_img_patches = []
for idx in range(imgs.shape[0]):  # imgs: (13, 898, 5552, 3)
    large_image = imgs[idx]

    # patchify expects a 3D patch shape for RGB
    patches_img = patchify(large_image, (patch_size, patch_size, 3), step=step)
    # patches_img shape: (nH, nW, 1, patch, patch, 3)

    for i in range(patches_img.shape[0]):
        for j in range(patches_img.shape[1]):
            single_patch_img = patches_img[i, j, 0, :, :, :]   # drop the singleton dim
            all_img_patches.append(single_patch_img)

images = np.array(all_img_patches, dtype=np.uint8)  # (num_patches, 256, 256, 3)
print("images patches:", images.shape, images.dtype)


# --- Masks (0,1,2) -> binary foreground/background ---
all_mask_patches = []
for idx in range(msks.shape[0]):  # msks: (13, 898, 5552)
    large_mask = msks[idx]

    patches_mask = patchify(large_mask, (patch_size, patch_size), step=step)
    # patches_mask shape: (nH, nW, patch, patch)

    for i in range(patches_mask.shape[0]):
        for j in range(patches_mask.shape[1]):
            single_patch_mask = patches_mask[i, j, :, :]

            # Convert 0/1/2 -> binary 0/1 (SAM-friendly default)
            single_patch_mask = (single_patch_mask > 0).astype(np.uint8)

            all_mask_patches.append(single_patch_mask)

masks = np.array(all_mask_patches, dtype=np.uint8)  # (num_patches, 256, 256)
print("mask patches:", masks.shape, masks.dtype)
print("mask unique:", np.unique(masks))


In [ ]:
#remove empty patches for faster training

keep = masks.reshape(masks.shape[0], -1).mean(axis=1) > 0.0005  # adjust threshold if needed
images = images[keep]
masks  = masks[keep]
print("kept patches:", images.shape[0])


In [ ]:
# visual checkpoint

import matplotlib.pyplot as plt
import random

k = random.randrange(images.shape[0])
plt.figure(figsize=(12,4))
plt.subplot(1,3,1); plt.title("Image"); plt.imshow(images[k]); plt.axis("off")
plt.subplot(1,3,2); plt.title("Mask"); plt.imshow(masks[k], cmap="gray"); plt.axis("off")
plt.subplot(1,3,3); plt.title("Overlay")
plt.imshow(images[k]); plt.imshow(masks[k], cmap="jet", alpha=0.35); plt.axis("off")
plt.show()


In [ ]:
valid_indices = [i for i, mask in enumerate(masks) if mask.max() != 0]
filtered_images = images[valid_indices]
filtered_masks  = masks[valid_indices]

print("Image shape:", filtered_images.shape)
print("Mask shape:", filtered_masks.shape)
#just another checkpoint don't mind

In [ ]:
from datasets import Dataset
from PIL import Image
import numpy as np

dataset_dict = {
    "image": [Image.fromarray(img, mode="RGB") for img in filtered_images],
    "label": [
        Image.fromarray((mask * 255).astype(np.uint8), mode="L")
        for mask in filtered_masks
    ],
}

dataset = Dataset.from_dict(dataset_dict)

print(dataset)


In [ ]:
# another visual sanity check

import matplotlib.pyplot as plt
import random

i = random.randrange(len(dataset))
plt.figure(figsize=(8,4))
plt.subplot(1,2,1)
plt.title("Image")
plt.imshow(dataset[i]["image"])
plt.axis("off")

plt.subplot(1,2,2)
plt.title("Mask")
plt.imshow(dataset[i]["label"], cmap="gray")
plt.axis("off")
plt.show()


In [ ]:
# next bounding boxes for promt training

import numpy as np

# Get bounding boxes from mask (robust)
def get_bounding_box(ground_truth_map, max_jitter=20):
    """
    ground_truth_map: (H,W) uint8 mask with foreground > 0
    returns: [x_min, y_min, x_max, y_max] or None if empty
    """
    ys, xs = np.where(ground_truth_map > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None  # empty mask safety

    x_min, x_max = xs.min(), xs.max()
    y_min, y_max = ys.min(), ys.max()

    H, W = ground_truth_map.shape

    # add random jitter
    x_min = max(0, x_min - np.random.randint(0, max_jitter + 1))
    x_max = min(W - 1, x_max + np.random.randint(0, max_jitter + 1))
    y_min = max(0, y_min - np.random.randint(0, max_jitter + 1))
    y_max = min(H - 1, y_max + np.random.randint(0, max_jitter + 1))

    return [int(x_min), int(y_min), int(x_max), int(y_max)]


In [ ]:
# sanity check

bbox = get_bounding_box(filtered_masks[0])
print("bbox:", bbox)


In [ ]:
# Custom PyTorch Dataset for SAM fine-tuning.
# For each sample, this dataset:
# 1) Loads an RGB image and its corresponding ground-truth segmentation mask
# 2) Derives a bounding-box prompt from the mask (used as input to SAM)
# 3) Uses the SAM processor to prepare the image and prompt for the model
# 4) Returns the processed model inputs together with the ground-truth mask,
#    which is later used to compute the segmentation loss during training

import numpy as np
import torch
from torch.utils.data import Dataset

class SAMDataset(Dataset):

    def __init__(self, dataset, processor):
        self.dataset = dataset
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item["image"]
        ground_truth_mask = np.array(item["label"])

        # make binary mask
        ground_truth_mask = (ground_truth_mask > 0).astype(np.uint8)

        # get bounding box prompt
        prompt = get_bounding_box(ground_truth_mask)

        # prepare image and prompt for the model
        inputs = self.processor(
            image,
            input_boxes=[[prompt]],
            return_tensors="pt"
        )

        # remove batch dimension
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}

        #  THIS IS THE IMPORTANT PART
        inputs["input_boxes"] = inputs["input_boxes"].float()
        inputs["ground_truth_mask"] = torch.from_numpy(
            ground_truth_mask
        ).unsqueeze(0).float()

        return inputs


In [ ]:
checkpoint = "facebook/sam-vit-base"
# checkpoint = "facebook/sam-vit-large"
# checkpoint = "facebook/sam-vit-huge"


from transformers import SamProcessor, SamModel
import torch

processor = SamProcessor.from_pretrained(checkpoint)

model = SamModel.from_pretrained(checkpoint)
model.to("cuda" if torch.cuda.is_available() else "cpu")





sam_dataset = SAMDataset(dataset, processor)
sample = sam_dataset[0]

for k, v in sample.items():
    if hasattr(v, "shape"):
        print(k, v.shape, v.dtype)
    else:
        print(k, type(v))


In [ ]:
# Initialize the processor
from transformers import SamProcessor
processor = SamProcessor.from_pretrained("facebook/sam-vit-base")

# Create an instance of the SAMDataset
train_dataset = SAMDataset(dataset=dataset, processor=processor)

# Inspect one example
example = train_dataset[0]
for k, v in example.items():
    if hasattr(v, "shape"):
        print(k, v.shape, v.dtype)
    else:
        print(k, type(v))

# Create a DataLoader for training
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    drop_last=False
)

# Inspect one batch
batch = next(iter(train_dataloader))
for k, v in batch.items():
    print(k, v.shape, v.dtype)

print("ground_truth_mask shape:", batch["ground_truth_mask"].shape)


In [ ]:
# training loop
# Load the model
from transformers import SamModel
import torch
import monai
from torch.optim import Adam
from tqdm import tqdm
from statistics import mean
import torch.nn.functional as F
import os

model = SamModel.from_pretrained("facebook/sam-vit-base")

# Only train mask decoder (freeze vision + prompt encoders)
for name, param in model.named_parameters():
    if name.startswith("vision_encoder") or name.startswith("prompt_encoder"):
        param.requires_grad_(False)

# Optimizer + loss
optimizer = Adam(model.mask_decoder.parameters(), lr=1e-5, weight_decay=0)
seg_loss = monai.losses.DiceCELoss(sigmoid=True, squared_pred=True, reduction="mean")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.train()

num_epochs = 1

for epoch in range(num_epochs):
    epoch_losses = []

    for batch in tqdm(train_dataloader):
        pixel_values = batch["pixel_values"].to(device)
        input_boxes  = batch["input_boxes"].to(device).float()        # (B,1,4) float32
        gt_masks     = batch["ground_truth_mask"].to(device).float()  # (B,1,256,256)

        # Forward pass
        outputs = model(
            pixel_values=pixel_values,
            input_boxes=input_boxes,
            multimask_output=False
        )

        # Predicted masks (logits). Different HF versions may add singleton dims.
        predicted_masks = outputs.pred_masks

        # ---- FIX SHAPES to (B, 1, H, W) ----
        # Possible shapes:
        # (B, H, W)           -> add channel
        # (B, 1, H, W)        -> ok
        # (B, 1, 1, H, W)     -> squeeze the extra singleton mask axis
        if predicted_masks.dim() == 3:
            predicted_masks = predicted_masks.unsqueeze(1)            # (B,H,W) -> (B,1,H,W)
        elif predicted_masks.dim() == 5 and predicted_masks.shape[2] == 1:
            predicted_masks = predicted_masks.squeeze(2)              # (B,1,1,H,W) -> (B,1,H,W)

        # Resize prediction to GT size if needed
        if predicted_masks.shape[-2:] != gt_masks.shape[-2:]:
            predicted_masks = F.interpolate(
                predicted_masks,
                size=gt_masks.shape[-2:],
                mode="bilinear",
                align_corners=False
            )

        # Compute loss (both are (B,1,256,256))
        loss = seg_loss(predicted_masks, gt_masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_losses.append(loss.item())

    print(f"EPOCH: {epoch}")
    print(f"Mean loss: {mean(epoch_losses):.6f}")

# Save checkpoint to Drive
save_dir = "/content/drive/MyDrive/Colab/models/SAM"
os.makedirs(save_dir, exist_ok=True)
ckpt_path = f"{save_dir}/blood_model_checkpoint.pth"
torch.save(model.state_dict(), ckpt_path)
print("Saved to:", ckpt_path)


This is the end of the training pipeline, next part is the application part


In [ ]:
#loading model, just change name to whichever model you wanna use

from transformers import SamModel, SamProcessor
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# Same checkpoint as you trained from
base_checkpoint = "facebook/sam-vit-base"

# Load processor + model architecture
processor = SamProcessor.from_pretrained(base_checkpoint)
model = SamModel.from_pretrained(base_checkpoint).to(device)

# Load your fine-tuned weights
ckpt_path = "/content/drive/MyDrive/Colab/models/SAM/blood_model_checkpoint.pth"
state_dict = torch.load(ckpt_path, map_location=device)
model.load_state_dict(state_dict)

model.eval()
print(" Loaded fine-tuned model from:", ckpt_path)


In [ ]:
import numpy as np
import random
import torch
import matplotlib.pyplot as plt

# pick a random example from the HF dataset
idx = random.randint(0, len(dataset) - 1)

# load image + mask from dataset (PIL)
test_image = dataset[idx]["image"]
ground_truth_mask = np.array(dataset[idx]["label"])

# make sure mask is binary 0/1 for bounding box extraction
ground_truth_mask_bin = (ground_truth_mask > 0).astype(np.uint8)

# get box prompt from mask
prompt = get_bounding_box(ground_truth_mask_bin)

# prepare image + box prompt for the model
inputs = processor(test_image, input_boxes=[[prompt]], return_tensors="pt")

# move to device
inputs = {k: v.to(device) for k, v in inputs.items()}
inputs["input_boxes"] = inputs["input_boxes"].float()  # keep float32

# use your loaded fine-tuned model:
# If you followed the load snippet I gave: it's called `model`
model.eval()

with torch.no_grad():
    outputs = model(**inputs, multimask_output=False)

# predicted mask logits -> probability
pred_logits = outputs.pred_masks

# normalize shape to (H,W)
# possible shapes: (1,1,H,W) or (1,1,1,H,W)
if pred_logits.dim() == 5 and pred_logits.shape[2] == 1:
    pred_logits = pred_logits.squeeze(2)

pred_prob = torch.sigmoid(pred_logits).squeeze().detach().cpu().numpy()  # (H,W)

# hard mask
pred_mask = (pred_prob > 0.5).astype(np.uint8)

# Plot
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

axes[0].imshow(np.array(test_image))
axes[0].set_title("Image")

axes[1].imshow(ground_truth_mask_bin, cmap="gray")
axes[1].set_title("Ground Truth Mask")

axes[2].imshow(pred_mask, cmap="gray")
axes[2].set_title("Predicted Mask")

axes[3].imshow(pred_prob)
axes[3].set_title("Probability Map")

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

plt.show()


In [ ]:
#loading model incase

from transformers import SamModel, SamProcessor
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

base_checkpoint = "facebook/sam-vit-base"
processor = SamProcessor.from_pretrained(base_checkpoint)

model = SamModel.from_pretrained(base_checkpoint).to(device)
ckpt_path = "/content/drive/MyDrive/ColabNotebooks/models/SAM/blood_model_checkpoint.pth"
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.eval()
print("model loaded")


In [ ]:
from PIL import Image
import numpy as np

img = Image.open("/content/drive/MyDrive/Colab/test_neu.png").convert("RGB")
new_img = np.array(img)

print(new_img.shape, new_img.dtype)


In [ ]:
print("new_img shape:", new_img.shape)  # should be (H, W, 3)
H, W = new_img.shape[:2]
print("H,W:", H, W)
img=new_img

In [ ]:
import numpy as np
from PIL import Image
import torch
import matplotlib.pyplot as plt

def pad_to_multiple(img, patch_size=256):
    H, W = img.shape[:2]
    Hp = int(np.ceil(H / patch_size) * patch_size)
    Wp = int(np.ceil(W / patch_size) * patch_size)
    padded = np.pad(img, ((0, Hp - H), (0, Wp - W), (0, 0)), mode="constant", constant_values=0)
    return padded, (H, W)

def predict_patch_prob_full_box(patch_rgb_uint8, model, processor, device):
    pil_patch = Image.fromarray(patch_rgb_uint8).convert("RGB")
    full_box = [[[0, 0, patch_rgb_uint8.shape[1]-1, patch_rgb_uint8.shape[0]-1]]]  # CPU list
    inputs = processor(pil_patch, input_boxes=full_box, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    inputs["input_boxes"] = inputs["input_boxes"].float()

    with torch.no_grad():
        outputs = model(**inputs, multimask_output=False)

    logits = outputs.pred_masks
    if logits.dim() == 5 and logits.shape[2] == 1:
        logits = logits.squeeze(2)

    prob = torch.sigmoid(logits).squeeze().detach().cpu().numpy().astype(np.float32)
    return prob

def predict_full_prob_overlap(img_rgb_uint8, model, processor, device,
                             patch_size=256, stride=128):
    padded, (H, W) = pad_to_multiple(img_rgb_uint8, patch_size)
    Hp, Wp = padded.shape[:2]

    prob_sum = np.zeros((Hp, Wp), dtype=np.float32)
    prob_cnt = np.zeros((Hp, Wp), dtype=np.float32)

    model.eval()
    for y0 in range(0, Hp - patch_size + 1, stride):
        for x0 in range(0, Wp - patch_size + 1, stride):
            patch = padded[y0:y0+patch_size, x0:x0+patch_size, :]
            prob = predict_patch_prob_full_box(patch, model, processor, device)

            prob_sum[y0:y0+patch_size, x0:x0+patch_size] += prob
            prob_cnt[y0:y0+patch_size, x0:x0+patch_size] += 1.0

    prob_avg = prob_sum / np.maximum(prob_cnt, 1e-6)
    return prob_avg[:H, :W]

# ---- RUN ----
full_prob = predict_full_prob_overlap(img, model, processor, device, patch_size=256, stride=128)

thresh = 0.6
full_mask = (full_prob > thresh).astype(np.uint8)

plt.figure(figsize=(18,6))
plt.subplot(1,3,1); plt.title("Image"); plt.imshow(img); plt.axis("off")
plt.subplot(1,3,2); plt.title("Probability (blended)"); plt.imshow(full_prob); plt.axis("off")
plt.subplot(1,3,3); plt.title(f"Mask (thr={thresh})"); plt.imshow(full_mask, cmap="gray"); plt.axis("off")
plt.show()


In [ ]:
from PIL import Image
import numpy as np

out_path = "/content/drive/MyDrive/Colab/test_image_mask_clean.png"

# convert 0/1 → 0/255 for image storage
mask_to_save = (full_mask * 255).astype(np.uint8)

Image.fromarray(mask_to_save).save(out_path)
print("Saved mask to:", out_path)


In [ ]:
import numpy as np
import cv2

def to_0_255(mask):
    """Ensure mask is uint8 0/255."""
    m = mask.astype(np.uint8)
    if m.max() == 1:
        m = m * 255
    return m

def remove_small_black_islands(mask_bw, max_black_area=200):
    """
    Remove small BLACK islands (noise) on WHITE background.
    mask_bw: uint8 0/255 where 0=foreground, 255=background
    """
    m = to_0_255(mask_bw)

    # Make foreground=1 for connected components: black -> 1
    fg = (m == 0).astype(np.uint8) * 255

    num, labels, stats, _ = cv2.connectedComponentsWithStats(fg, connectivity=8)

    cleaned_fg = np.zeros_like(fg, dtype=np.uint8)
    for lab in range(1, num):
        area = stats[lab, cv2.CC_STAT_AREA]
        if area >= max_black_area:
            cleaned_fg[labels == lab] = 255

    # Convert back to mask convention: fg(255) -> black(0), bg -> white(255)
    out = np.full_like(m, 255, dtype=np.uint8)
    out[cleaned_fg == 255] = 0
    return out

def fill_small_white_holes_in_black(mask_bw, max_hole_area=200):
    """
    Fill small WHITE holes inside BLACK
    mask_bw: uint8 0/255 where 0=foreground, 255=background
    """
    m = to_0_255(mask_bw)

    # Invert so that holes (white inside black) become foreground (1)
    # Here, "white pixels" are 1s in inv, but we only want holes that are enclosed by black,
    # not the outside background. So: label white regions and fill those that DON'T touch border.
    holes = (m == 255).astype(np.uint8) * 255  # all white regions

    num, labels, stats, _ = cv2.connectedComponentsWithStats(holes, connectivity=8)

    H, W = m.shape
    out = m.copy()

    for lab in range(1, num):
        area = stats[lab, cv2.CC_STAT_AREA]
        if area > max_hole_area:
            continue

        x = stats[lab, cv2.CC_STAT_LEFT]
        y = stats[lab, cv2.CC_STAT_TOP]
        w = stats[lab, cv2.CC_STAT_WIDTH]
        h = stats[lab, cv2.CC_STAT_HEIGHT]

        touches_border = (x == 0) or (y == 0) or (x + w == W) or (y + h == H)

        # Only fill enclosed white regions (holes), not background
        if not touches_border:
            out[labels == lab] = 0  # fill hole with black (mito)

    return out

def morph_smooth(mask_bw, k_close=5, k_open=0):
    """
    Smooth boundaries for black-foreground masks.
    - close helps fill tiny white gaps inside black
    - open helps remove tiny black specks
    """
    m = to_0_255(mask_bw)

    # Convert black foreground to 255 to apply morphology on foreground
    fg = (m == 0).astype(np.uint8) * 255

    if k_open and k_open > 0:
        ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_open, k_open))
        fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, ker)

    if k_close and k_close > 0:
        ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_close, k_close))
        fg = cv2.morphologyEx(fg, cv2.MORPH_CLOSE, ker)

    # Convert back
    out = np.full_like(m, 255, dtype=np.uint8)
    out[fg == 255] = 0
    return out


# ============================
# APPLY CLEANUP (tune params)
# ============================
# full_mask currently is 0/1 or 0/255? We normalize:
mask_bw = to_0_255(full_mask)   # <- your current mask (black cluster, white background)

# 1) Remove small black noise islands
mask_bw = remove_small_black_islands(mask_bw, max_black_area=1500)  # try 50, 100, 300, 800...

# 2) Fill small white holes
mask_bw = fill_small_white_holes_in_black(mask_bw, max_hole_area=300)  # try 30, 100, 300, 800...

# 3) Optional smoothing
mask_bw = morph_smooth(mask_bw, k_close=5, k_open=0)

mask_clean_bw = mask_bw  # final (0=mito, 255=background)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(16,6))
plt.subplot(1,2,1); plt.title("Original mask (black=cliuster)"); plt.imshow(to_0_255(full_mask), cmap="gray"); plt.axis("off")
plt.subplot(1,2,2); plt.title("Cleaned mask"); plt.imshow(mask_clean_bw, cmap="gray"); plt.axis("off")
plt.show()





In [ ]:
from PIL import Image

out_path = "/content/drive/MyDrive/Colab/test_image_mask_clean_final.png"
Image.fromarray(mask_clean_bw).save(out_path)
print("Saved:", out_path)

